# Limpieza del data set 

In [1]:
import pandas as pd
file_name = 'Global Food & Nutrition Database 2026 – Health Scores & Allergens.csv'
df = pd.read_csv(file_name)

print(df.columns.tolist())
df.info()

['product_name', 'brands', 'categories', 'ingredients', 'nutriscore_grade', 'nova_group', 'ecoscore_grade', 'allergens', 'energy_kcal', 'fat_100g', 'saturated_fat_100g', 'carbs_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g', 'contains_gluten', 'contains_dairy', 'contains_nuts', 'contains_soy', 'contains_eggs', 'contains_fish', 'food_type']
<class 'pandas.DataFrame'>
RangeIndex: 4997 entries, 0 to 4996
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_name        4785 non-null   str    
 1   brands              4751 non-null   str    
 2   categories          4913 non-null   str    
 3   ingredients         4797 non-null   str    
 4   nutriscore_grade    4983 non-null   str    
 5   nova_group          4522 non-null   float64
 6   ecoscore_grade      4984 non-null   str    
 7   allergens           3332 non-null   str    
 8   energy_kcal         4682 non-null 

 Como analizamos en el informe, vemos que los datos como nova_group que puede tomar valores del 1 al 4 es un float64, y que columnas como nutriscore_grade y ecoscore_grade que van de la A a la E (variables cualitativas ordinales) son de tipo object. Transformaremos nova_group a un Int8 para conservar los NaN y las otras dos variables de tipo numéricas para poder trabajarlas correctamente más tarde, ya que casi todas las herramientas de análisis estadístico avanzado en Python lo exigen

In [2]:
df['nova_group'] = df['nova_group'].astype('Int8')

In [3]:
mapeo_ordinal = {
    'A': 5,
    'B': 4,
    'C': 3,
    'D': 2,
    'E': 1
}

df['nutriscore_num'] = df['nutriscore_grade'].str.strip().str.upper().map(mapeo_ordinal).astype('Int8')
df['ecoscore_num'] = df['ecoscore_grade'].str.strip().str.upper().map(mapeo_ordinal).astype('Int8')

df = df.drop(columns=['nutriscore_grade', 'ecoscore_grade'])

In [4]:
## Posteriormente se puede aplicar los siguiente si se queiren visualizar correctamente los datos 
#import matplotlib.pyplot as plt

#df['nutriscore_num'].value_counts().sort_index().plot(kind='bar')

#plt.xticks(
#    ticks=range(5),
#    labels=['E', 'D', 'C', 'B', 'A'],
#    rotation=0
#)

#plt.xlabel('Nutri-Score')
#plt.ylabel('Cantidad de productos')
#plt.title('Distribución del Nutri-Score')
#plt.show()

El riesgo del falso NAN: 
El único problema con columnas de texto como product name es que el CSV puede tener literalmente la palabra escrita.

- Si la celda del CSV estaba vacía, pandas la lee como el objeto NaN y df.isnull() sí la elimina.

- Si la celda del CSV tenía escrita la palabra "nan", "NaN" o "null" como texto, pandas cree que ese es el nombre del producto. df.isnull() no lo va a contar porque es un String válido.

Para asegurarnos de que no vamos a tener ese problema: 

In [5]:
import numpy as np

falsos_nulos = ['nan', 'NaN', 'null', 'Null', '']

df = df.replace(falsos_nulos, np.nan)

print(df.isnull().sum())

product_name           212
brands                 246
categories              84
ingredients            200
nova_group             475
allergens             1665
energy_kcal            315
fat_100g               302
saturated_fat_100g     380
carbs_100g             321
sugars_100g            366
fiber_100g            1502
proteins_100g          307
salt_100g              344
sodium_100g            344
contains_gluten          0
contains_dairy           0
contains_nuts            0
contains_soy             0
contains_eggs            0
contains_fish            0
food_type                0
nutriscore_num         550
ecoscore_num          1413
dtype: int64


Sabemos que todos los productos son empaquetados, por lo que agregaremos a la metadata que solo estamos trabajando con productos de este tipo

In [6]:
# Verifica si absolutamente todos los valores coinciden con el string
if (df['food_type'].str.lower() == 'branded/packaged').all():
    df = df.drop(columns=['food_type'])
    print("Columna 'food type' eliminada: todos los registros eran 'Branded/packaged'.")
else:
    print("La columna 'food type' se conserva: contiene valores distintos a 'Branded/packaged'.")

Columna 'food type' eliminada: todos los registros eran 'Branded/packaged'.


Tambien debemos considerar eliminar valores mal cargados (fuera de rango)

In [7]:
registros_antes = len(df)

columnas_100g = [col for col in df.columns if col.endswith('_100g')]

df = df[
    (
        df[columnas_100g].ge(0) |
        df[columnas_100g].isna()
    ).all(axis=1) &
    (
        df[columnas_100g].le(100) |
        df[columnas_100g].isna()
    ).all(axis=1)
]

registros_despues = len(df)

print(f"Registros originales: {registros_antes}")
print(f"Registros finales: {registros_despues}")
print(f"Registros eliminados: {registros_antes - registros_despues}")

Registros originales: 4997
Registros finales: 4991
Registros eliminados: 6


Acerca de los REPETIDOS:

In [8]:
cant_duplicados = df.duplicated().sum()
print("cantidad total de duplicados: " , cant_duplicados)

cantidad total de duplicados:  39


In [9]:
columnas_a_ignorar = ['Name', 'Brand']

columnas_atributos = df.columns.difference(columnas_a_ignorar)

duplicados_por_atributos = df.duplicated(subset=columnas_atributos).sum()
print(f"Duplicados ignorando nombre y marca: {duplicados_por_atributos}")

Duplicados ignorando nombre y marca: 39


En principio pareceria que tenemos solo 38 filas duplicadas realmente, ya que nos fijamos en este ejemplo en todas las columnas excepto nombre y marca que podrían traer problemas por el idioma

In [10]:

columnas_a_ignorar = ['Name', 'Brand','categories', 'ingredients','allergens']

columnas_atributos = df.columns.difference(columnas_a_ignorar)

duplicados_por_atributos = df.duplicated(subset=columnas_atributos).sum()
print(f"Duplicados ignorando nombre, marca, categorias, ingredientes y alergenos: {duplicados_por_atributos}")

filas_duplicadas = df[df.duplicated(subset=columnas_atributos, keep=False)]

duplicados_agrupados = filas_duplicadas.sort_values(by=list(columnas_atributos))
print(f"Cantidad de filas antes: {len(df)}")

with pd.option_context('display.max_rows', None):
    display(duplicados_agrupados)
    
# para una visualización mas sencilla, usar la linea de abajo y comentar lo de arriba
# display(duplicados_agrupados)

Duplicados ignorando nombre, marca, categorias, ingredientes y alergenos: 74
Cantidad de filas antes: 4991


,product_name,brands,categories,ingredients,nova_group,allergens,energy_kcal,fat_100g,saturated_fat_100g,carbs_100g,...,salt_100g,sodium_100g,contains_gluten,contains_dairy,contains_nuts,contains_soy,contains_eggs,contains_fish,nutriscore_num,ecoscore_num
87,Pâte à tartiner noisettes et cacao,Bonne Maman,"en:breakfasts, en:spreads, en:sweet-spreads","sucre, _noisettes_ 20%, huiles végétales (tour...",4,"en:milk, en:nuts",551.0,34.00,5.2,53.0,...,0.10,0.040,False,True,True,False,False,False,1,2
516,Pâte à tartiner noisettes et cacao,Bonne Maman,"en:breakfasts, en:spreads, en:sweet-spreads","Sucre, NOISETTES 20%, huiles végétales (tourne...",4,"en:milk, en:nuts",551.0,34.00,5.2,53.0,...,0.10,0.040,False,True,True,False,False,False,1,2
3543,Petits pains grilles,Carrefour,"en:plant-based-foods-and-beverages, en:plant-b...","Farine de _blé_ 46%, farine de _blé_ complet 4...",4,en:gluten,408.0,10.00,0.9,66.0,...,1.00,0.400,True,False,False,False,False,False,4,5
4025,Petits pains grilles,Carrefour,"en:plant-based-foods-and-beverages, en:plant-b...","Farine de _blé_ 46%, farine de _blé_ complet 4...",4,en:gluten,408.0,10.00,0.9,66.0,...,1.00,0.400,True,False,False,False,False,False,4,5
400,Ratatouille Cuisinée à la Provençal,Cassegrain,"en:plant-based-foods-and-beverages, en:plant-b...",Dés d'aubergines préfrites 25% (aubergines 22%...,3,en:gluten,72.0,4.10,0.6,5.7,...,0.81,0.324,True,False,False,False,False,False,3,4
955,Ratatouille Cuisinée à la Provençal,Cassegrain,"en:plant-based-foods-and-beverages, en:plant-b...",Dés d'aubergines préfrites 25% (aubergines 22%...,3,"en:gluten, en:cereals-with-gluten",72.0,4.10,0.6,5.7,...,0.81,0.324,True,False,False,False,False,False,3,4
546,Our Mature Cheddar,Cathedral City,"en:dairies, en:fermented-foods, en:fermented-m...",cheddar. contains milk.,3,en:milk,416.0,34.90,21.7,0.1,...,1.77,0.708,False,True,False,False,False,False,2,2
1419,Our Mature Cheddar,Cathedral City,"en:dairies, en:fermented-foods, en:fermented-m...",Cheddar Cheese. Contains Milk.,3,en:milk,416.0,34.90,21.7,0.1,...,1.77,0.708,False,True,False,False,False,False,2,2
956,Coca-Cola Zero,Coca-Cola,"en:beverages-and-beverages-preparations, en:be...","Wasser, Kohlensäure, Farbstoff E 150d, Süssung...",4,NaN,0.3,0.00,0.0,0.0,...,0.01,0.004,False,False,False,False,False,False,3,<NA>
3554,Coca-Cola Zero,Coca-Cola,"en:beverages-and-beverages-preparations, en:be...","Agua carbonatada, Colorante e-150d, Edulcorant...",4,en:phenylalanine,0.3,0.00,0.0,0.0,...,0.01,0.004,False,False,False,False,False,False,3,<NA>


Sin embargo vemos que tenemos muchos repetidos que no eran tenidos en cuenta porque sus ingredientes estaban en dos idiomas distintos o algunos tenían mayúsculas y otros no. Además, vemos una gran cantidad de productos que solo tienen atributos como los ingredientes o sus categorías y no aportan nada acerca de su valor nutricional; por lo que también los descartaremos 

In [11]:
# Eliminar los duplicados basados en los atributos, conservando la primera aparición
df = df.drop_duplicates(subset=columnas_atributos, keep='first')

# Verificar la cantidad de filas que quedaron en el dataset
print(f"Cantidad de filas tras eliminar duplicados: {len(df)}")

Cantidad de filas tras eliminar duplicados: 4917


In [12]:
filas_originales = len(df)

macros_principales = [
    'energy_kcal', 
    'fat_100g', 
    'saturated_fat_100g', 
    'carbs_100g', 
    'sugars_100g', 
    'fiber_100g', 
    'proteins_100g', 
    'salt_100g', 
    'sodium_100g'
]

# se exige al menos 3 datos válidos de los 9
df = df.dropna(subset=macros_principales, thresh=3)

filas_eliminadas = filas_originales - len(df)
print(f"Filas eliminadas por falta de datos nutricionales: {filas_eliminadas}")
print(f"Filas restantes en el dataset: {len(df)}")

Filas eliminadas por falta de datos nutricionales: 231
Filas restantes en el dataset: 4686


También es interesante plantearse la opción de agregar una columna para poder filtrar la mayor cantidad posible de alimentos segun a que categoría pertenecen, evitando generar sesgos (por ejemplo, contar grasas de productos que en realidad son bebidas y no tienen grasas)

In [13]:
import numpy as np

cats_lower = df['categories'].astype(str).str.lower()

condiciones = [
    # Bebidas (Excluye 'foods-and-beverages' para evitar meter frutos secos y cereales)
    cats_lower.str.contains(r'\b(?:(?<!foods-and-)beverages?|boissons?|bebidas?|juices?|jus|sodas?|waters?|eaux?|aguas?|teas?|thés?|coffees?|cafés?)\b', regex=True),
    
    # Lácteos y derivados
    cats_lower.str.contains(r'\b(?:dairy|dairies|laits?|cheeses?|fromages?|yogurts?|yaourts?|milks?|lacteos?|quesos?)\b', regex=True),
    
    # Snacks dulces, postres y repostería
    cats_lower.str.contains(r'\b(?:snacks?|biscuits?|cakes?|chocolates?|chocolat|cocoa|sweets?|desserts?|biscottes?|confiseries?|galletas?|dulces?)\b', regex=True),
    
    # Comidas preparadas, platos listos y congelados
    cats_lower.str.contains(r'\b(?:meals?|pizzas?|pies?|quiches?|prepared|ready-to-eat|plat-prepare|dishes|dish|salads?|frozen-foods?)\b', regex=True),
    
    # Caldos, sopas, bases para cocinar y deshidratados
    cats_lower.str.contains(r'\b(?:broths?|bouillons?|soups?|soupes?|dried-product|cooking-helper|sopas?|caldos?)\b', regex=True),
    
    # Untables (dulces o salados)
    cats_lower.str.contains(r'\b(?:spreads?|tartines?|pate|paté|pastes?|yeast-extract)\b', regex=True),
    
    # Alimentos a base de plantas, cereales, frutas y verduras
    cats_lower.str.contains(r'\b(?:plant-based|cereals?|breads?|pain|pan|potatoes|potato|legumes?|vegetables?|fruits?|frutas?)\b', regex=True),
    
    # Carnes, pescados, huevos y mariscos
    cats_lower.str.contains(r'\b(?:fish|seafood|meats?|poissons?|viandes?|poultry|pisciculture|eggs?|oeufs?|huevos?|pescados?|carnes?)\b', regex=True),
    
    # Aceites, grasas, salsas y condimentos
    cats_lower.str.contains(r'\b(?:oils?|huiles?|aceites?|sauces?|condiments?|fats?|matiere-grasse)\b', regex=True),
    
    # Suplementos nutricionales y aditivos
    cats_lower.str.contains(r'\b(?:supplements?|dietary|meal-replacement|additives?|yeast)\b', regex=True)
]

categorias_macro = [
    'Bebidas',
    'Lácteos',
    'Snacks y Dulces',
    'Comidas Preparadas',
    'Caldos y Sopas',
    'Untables',
    'Vegetales y Cereales',
    'Carnes y Pescados',
    'Grasas y Salsas',
    'Suplementos y Aditivos'
]

# np.select evalúa en orden. Lo que coincida primero, se asigna.
df['categoria_macro'] = np.select(condiciones, categorias_macro, default='Otros')

# Verificar el nuevo recuento
print(df['categoria_macro'].value_counts())

categoria_macro
Vegetales y Cereales      1499
Snacks y Dulces           1163
Lácteos                    672
Bebidas                    621
Grasas y Salsas            257
Comidas Preparadas         193
Carnes y Pescados          132
Otros                      100
Untables                    22
Caldos y Sopas              14
Suplementos y Aditivos      13
Name: count, dtype: int64


In [14]:
# rellenamos el tratado de las columnas de datos que aportan vbalor energético por cero en lugar de Nan 
columnas = ['energy_kcal', 'fat_100g', 'saturated_fat_100g', 'carbs_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g']
df[columnas] = df[columnas].fillna(0)

In [17]:
# Guardar dataset limpio
df.to_parquet(
    'dataset_limpio.parquet',
    index=False,
    engine='pyarrow'
)

ArrowKeyError: A type extension with name pandas.period already defined

# Hipotesis 1 planteada por nosotros


In [ ]:
#### Test para la hipotesis 1 planteada por nosotros#####


import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_parquet("dataset_limpio.parquet")

# 1. Limpieza y preparación de la variable de interés
sodio_clean = df['sodium_100g'].dropna()

# 2. Definición del umbral normativo (0.3g = 300mg)
mu_0 = 0.3

# 3. Análisis Descriptivo Univariado
n_obs = len(sodio_clean)
media = sodio_clean.mean()
mediana = sodio_clean.median()
std = sodio_clean.std()
iqr = stats.iqr(sodio_clean)

print("--- ANÁLISIS DESCRIPTIVO ---")
print(f"N observaciones: {n_obs}")
print(f"Media: {media:.4f} g/100g")
print(f"Mediana: {mediana:.4f} g/100g")
print(f"Desviación Estándar: {std:.4f}")
print(f"Rango Intercuartílico (IQR): {iqr:.4f}")

# 4. Prueba t de Student para una muestra (Unilateral Superior)
t_stat, p_val_t = stats.ttest_1samp(sodio_clean, popmean=mu_0, alternative='greater')

# 5. Complemento No Paramétrico (Prueba de Wilcoxon respecto a la mediana)
res_wilcoxon = stats.wilcoxon(sodio_clean - mu_0, alternative='greater')

print("\n--- RESULTADOS DE LAS PRUEBAS ---")
print(f"Prueba t - Estadístico t: {t_stat:.4f} | p-valor: {p_val_t:.5e}")
print(f"Prueba Wilcoxon - Estadístico: {res_wilcoxon.statistic:.4f} | p-valor: {res_wilcoxon.pvalue:.5e}")

# Hipotesis 2 planteada por nosotros

In [ ]:
###test de la hipotesis 2 planteada por nosotros###


import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_parquet("dataset_limpio.parquet")

# 1. Limpieza y preparación de las variables de interés
df_ecoscore = df[['contains_soy', 'ecoscore_num']].dropna().copy()
df_ecoscore['contains_soy'] = df_ecoscore['contains_soy'].astype(bool)

# 2. Crear la variable binaria de Ecoscore desfavorable (1 o 2 = D/E = 1, 3/4/5 = A/B/C = 0)
df_ecoscore['ecoscore_desfavorable'] = df_ecoscore['ecoscore_num'].isin([1, 2]).astype(int)

# 3. Segmentar los datos en los dos grupos
grupo_con_soja = df_ecoscore[df_ecoscore['contains_soy'] == True]['ecoscore_desfavorable']
grupo_sin_soja = df_ecoscore[df_ecoscore['contains_soy'] == False]['ecoscore_desfavorable']

# 4. Prueba Z de Dos Proporciones (Unilateral: proporción con soja > proporción sin soja)
exitos = [grupo_con_soja.sum(), grupo_sin_soja.sum()]
observaciones = [len(grupo_con_soja), len(grupo_sin_soja)]

z_stat, p_val_prop = proportions_ztest(count=exitos, nobs=observaciones, alternative='larger')

print("--- PRUEBA Z DE DOS PROPORCIONES (Ecoscore Desfavorable 1 y 2) ---")
print(f"Proporción con Soja (Desfavorable): {grupo_con_soja.mean():.4f}")
print(f"Proporción sin Soja (Desfavorable): {grupo_sin_soja.mean():.4f}")
print(f"Estadístico Z: {z_stat:.4f} | p-valor: {p_val_prop:.5e}")

# 5. Prueba Chi-cuadrado de Independencia (para toda la tabla de contingencia 1 a 5 vs Soja)
tabla_contingencia = pd.crosstab(df_ecoscore['contains_soy'], df_ecoscore['ecoscore_num'])
chi2, p_val_chi2, dof, ex = stats.chi2_contingency(tabla_contingencia)

print("\n--- PRUEBA CHI-CUADRADO (Distribución global Ecoscore) ---")
print(f"Estadístico Chi2: {chi2:.4f} | p-valor: {p_val_chi2:.5e}")

# 6. Prueba U de Mann-Whitney (Ordinal: 1=E ... 5=A)
# 'less' evalúa si el grupo con soja tiene valores significativamente menores (peores notas)
grupo_con_soja_num = df_ecoscore[df_ecoscore['contains_soy'] == True]['ecoscore_num']
grupo_sin_soja_num = df_ecoscore[df_ecoscore['contains_soy'] == False]['ecoscore_num']

u_stat, p_val_mann = stats.mannwhitneyu(
    grupo_con_soja_num,
    grupo_sin_soja_num,
    alternative='less'
)

print("\n--- PRUEBA U DE MANN-WHITNEY (Ordinal) ---")
print(f"Estadístico U: {u_stat:.4f} | p-valor: {p_val_mann:.5e}")

In [ ]:
####mapa de calor para la soja y el ecoscore####

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


df = pd.read_parquet("dataset_limpio.parquet")


# 1. Preparar los datos (filtrar nulos)
df_heatmap_eco = df[['contains_soy', 'ecoscore_num']].dropna().copy()

# Mapeo de etiquetas claras para el gráfico
df_heatmap_eco['Soja'] = df_heatmap_eco['contains_soy'].map(
    {True: 'Con Soja', False: 'Sin Soja'}
)

# Mapeo ordinal: 1=E (peor impacto) a 5=A (mejor impacto)
eco_map = {1: 'E (1)', 2: 'D (2)', 3: 'C (3)', 4: 'B (4)', 5: 'A (5)'}
df_heatmap_eco['Ecoscore'] = df_heatmap_eco['ecoscore_num'].map(eco_map)

# 2. Crear tabla de contingencia en porcentajes (normalizada por filas)
ct_ecoscore = (
    pd.crosstab(
        df_heatmap_eco['Soja'], df_heatmap_eco['Ecoscore'], normalize='index'
    )
    * 100
)

# 3. Graficar el Mapa de Calor
plt.figure(figsize=(9, 5))
sns.heatmap(
    ct_ecoscore,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    cbar_kws={'label': 'Porcentaje (%) dentro del grupo'},
)

plt.title(
    'Distribución Porcentual de Ecoscore según Presencia de Soja',
    fontsize=13,
    pad=15,
)
plt.xlabel('Calificación Ecoscore', fontsize=11)
plt.ylabel('Presencia de Soja', fontsize=11)

plt.tight_layout()
plt.show()

# Hipotesis 3 planteada por nosotros

In [ ]:
import pandas as pd 

df = pd.read_parquet("dataset_limpio.parquet")

df_snacks = df[df["categoria_macro"] == "Snacks y Dulces"].copy()

print(len(df_snacks))
df_snacks.head()

In [ ]:
columnas = [
    'proteins_100g',
    'fiber_100g',
    'fat_100g',
    'energy_kcal'
]

for col in columnas:
    df_snacks[col] = pd.to_numeric(df_snacks[col], errors='coerce')

df_snacks[columnas].isnull().sum()


Vamos a crear dos clasificaciones independientes:
- Proteínas: productos con contenido elevado de proteínas frente al resto.
- Fibra: productos con contenido elevado de fibra frente al resto.
Utilizamos el percentil 75 para definir el contenido elevado.

In [ ]:
# Umbrales del percentil 75
umbral_proteinas = df_snacks['proteins_100g'].quantile(0.75)
umbral_fibra = df_snacks['fiber_100g'].quantile(0.75)

print("Umbral de proteínas:", umbral_proteinas)
print("Umbral de fibra:", umbral_fibra)

In [ ]:
import numpy as np

df_snacks['grupo_proteinas'] = np.where(
    df_snacks['proteins_100g'] >= umbral_proteinas,
    'Alto contenido',
    'Resto'
)

df_snacks['grupo_fibra'] = np.where(
    df_snacks['fiber_100g'] >= umbral_fibra,
    'Alto contenido',
    'Resto'
)

In [ ]:
print("Grupos de proteínas:")
print(df_snacks['grupo_proteinas'].value_counts())

print("\nGrupos de fibra:")
print(df_snacks['grupo_fibra'].value_counts())

In [ ]:
variables_resultado = ['fat_100g', 'energy_kcal']

for grupo in ['grupo_proteinas', 'grupo_fibra']:

    print(f"\nClasificación: {grupo}")

    for variable in variables_resultado:

        resumen = df_snacks.groupby(grupo)[variable].agg(
            ['count', 'mean', 'median', 'std']
        )

        print(f"\nVariable: {variable}")
        display(resumen)

In [ ]:
from scipy import stats

def comprobar_supuestos(df, variable_grupo, variable_resultado):

    datos = df[[variable_grupo, variable_resultado]].dropna()

    grupo_alto = datos.loc[
        datos[variable_grupo] == 'Alto contenido',
        variable_resultado
    ]

    grupo_resto = datos.loc[
        datos[variable_grupo] == 'Resto',
        variable_resultado
    ]

    # Shapiro-Wilk: admite entre 3 y 5000 observaciones
    def shapiro_seguro(x):
        if len(x) < 3:
            return np.nan

        if len(x) > 5000:
            x = x.sample(5000, random_state=42)

        return stats.shapiro(x).pvalue

    p_normal_alto = shapiro_seguro(grupo_alto)
    p_normal_resto = shapiro_seguro(grupo_resto)

    # Levene: comparación de varianzas
    p_levene = stats.levene(
        grupo_alto,
        grupo_resto,
        center='median'
    ).pvalue

    print(f"Variable analizada: {variable_resultado}")
    print(f"Grupo alto: n = {len(grupo_alto)}")
    print(f"Resto: n = {len(grupo_resto)}")

    print("\nNormalidad - Shapiro-Wilk")
    print(f"Alto contenido: p = {p_normal_alto:.4f}")
    print(f"Resto: p = {p_normal_resto:.4f}")

    print("\nHomocedasticidad - Levene")
    print(f"p = {p_levene:.4f}")

    return {
        'grupo_alto': grupo_alto,
        'grupo_resto': grupo_resto,
        'p_normal_alto': p_normal_alto,
        'p_normal_resto': p_normal_resto,
        'p_levene': p_levene
    }

In [ ]:
resultado = comprobar_supuestos(
    df_snacks,
    'grupo_proteinas',
    'fat_100g'
)

In [ ]:
def comparar_grupos(resultado):

    grupo_alto = resultado['grupo_alto']
    grupo_resto = resultado['grupo_resto']

    p_normal_alto = resultado['p_normal_alto']
    p_normal_resto = resultado['p_normal_resto']
    p_levene = resultado['p_levene']

    # Comprobar normalidad
    normalidad = (
        p_normal_alto > 0.05 and
        p_normal_resto > 0.05
    )

    if normalidad:

        if p_levene > 0.05:

            nombre_test = 't de Student'

            estadistico, p_valor = stats.ttest_ind(
                grupo_alto,
                grupo_resto,
                equal_var=True
            )

        else:

            nombre_test = 't de Welch'

            estadistico, p_valor = stats.ttest_ind(
                grupo_alto,
                grupo_resto,
                equal_var=False
            )

    else:

        nombre_test = 'Mann-Whitney U'

        estadistico, p_valor = stats.mannwhitneyu(
            grupo_alto,
            grupo_resto,
            alternative='two-sided'
        )

    diferencia_medias = grupo_alto.mean() - grupo_resto.mean()
    diferencia_medianas = grupo_alto.median() - grupo_resto.median()

    print("Test seleccionado:", nombre_test)
    print("Estadístico:", round(estadistico, 4))
    print("P-valor:", round(p_valor, 6))

    print("\nDiferencias descriptivas:")
    print("Diferencia de medias:", round(diferencia_medias, 2))
    print("Diferencia de medianas:", round(diferencia_medianas, 2))

    if p_valor < 0.05:
        print("\nResultado: existen diferencias estadísticamente significativas.")
    else:
        print("\nResultado: no se encontraron diferencias estadísticamente significativas.")

    return {
        'test': nombre_test,
        'estadistico': estadistico,
        'p_valor': p_valor,
        'diferencia_medias': diferencia_medias,
        'diferencia_medianas': diferencia_medianas
    }

In [ ]:
comparaciones = [
    ('grupo_proteinas', 'fat_100g'),
    ('grupo_proteinas', 'energy_kcal'),
    ('grupo_fibra', 'fat_100g'),
    ('grupo_fibra', 'energy_kcal')
]

resultados = []

for grupo, variable in comparaciones:

    print("\n" + "="*60)
    print(f"Comparación: {grupo} vs. {variable}")
    print("="*60)

    supuestos = comprobar_supuestos(
        df_snacks,
        grupo,
        variable
    )

    test = comparar_grupos(supuestos)

    resultados.append({
        'grupo': grupo,
        'variable': variable,
        **test
    })

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


umbral_proteinas = df_snacks["proteins_100g"].quantile(0.75)
umbral_fibra = df_snacks["fiber_100g"].quantile(0.75)

df_snacks["grupo_proteinas"] = np.where(
    df_snacks["proteins_100g"] >= umbral_proteinas,
    "Alto contenido",
    "Resto",
)
df_snacks["grupo_fibra"] = np.where(
    df_snacks["fiber_100g"] >= umbral_fibra, "Alto contenido", "Resto"
)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

graficos = [
    ("grupo_proteinas", "fat_100g", "Proteínas vs. Grasas"),
    ("grupo_proteinas", "energy_kcal", "Proteínas vs. Energía"),
    ("grupo_fibra", "fat_100g", "Fibra vs. Grasas"),
    ("grupo_fibra", "energy_kcal", "Fibra vs. Energía"),
]

for ax, (grupo, variable, titulo) in zip(axes.flat, graficos):
    sns.boxplot(data=df_snacks, x=grupo, y=variable, ax=ax, legend=False,hue=grupo, palette="Set2")

    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.set_ylabel(variable)

plt.suptitle("Comparación nutricional de snacks", fontsize=14)
plt.tight_layout()
plt.show()

# Hipotesis 1 planteada por la catedra 


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

df = pd.read_parquet('dataset_limpio.parquet')
df.head()

In [ ]:
grasas_saturadas = df['saturated_fat_100g'].dropna()

altos_en_grasa = grasas_saturadas > 5

exitos = altos_en_grasa.sum()         
n_observaciones = len(grasas_saturadas) 
proporcion_muestral = exitos / n_observaciones

print(f"Productos altos en grasas saturadas: {exitos}")
print(f"Total de observaciones válidas: {n_observaciones}")
print(f"Proporción en la muestra: {proporcion_muestral:.4f} ({proporcion_muestral*100:.2f}%)")

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

p_0 = 0.25 

z_stat, p_valor = proportions_ztest(count=exitos, nobs=n_observaciones, value=p_0, alternative='larger')

print(f"Estadístico Z: {z_stat:.4f}")
print(f"P-valor: {p_valor:.4e}")

alpha = 0.05
if p_valor < alpha:
    print("\nConclusión: Se rechaza la Hipótesis Nula (H0).")
    print("Hay evidencia estadística para afirmar que la proporción de productos altos en grasas saturadas es superior al 25%.")
else:
    print("\nConclusión: No se rechaza la Hipótesis Nula (H0).")
    print("No hay evidencia estadística suficiente para afirmar que la proporción supera el 25%.")

# Hipotesis 2 planteada por la catedra

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

df = pd.read_parquet('dataset_limpio.parquet')
df.head()

In [ ]:
import pandas as pd
from scipy import stats

df_4b = df[['nova_group', 'nutriscore_num']].dropna().copy()

tabla_contingencia = pd.crosstab(df_4b['nova_group'], df_4b['nutriscore_num'])
chi2, p_chi2, dof, expected = stats.chi2_contingency(tabla_contingencia)

coef_spearman, p_spearman = stats.spearmanr(df_4b['nova_group'], df_4b['nutriscore_num'])

print("--- PRUEBA DE INDEPENDENCIA CHI-CUADRADO ---")
print(f"Estadístico Chi2: {chi2:.4f}")
print(f"P-valor Chi2: {p_chi2:.4e}\n")

print("--- CORRELACIÓN DE SPEARMAN ---")
print(f"Coeficiente de Spearman: {coef_spearman:.4f}")
print(f"P-valor Spearman: {p_spearman:.4e}\n")

print("--- TABLA DE CONTINGENCIA (OBSERVACIONES REALES) ---")
print(tabla_contingencia)

# Hipotesis 3 planteada por la catedra 

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

df = pd.read_parquet('dataset_limpio.parquet')
df.head()

In [ ]:
if 'allergens_n' not in df.columns and 'allergens' in df.columns:
    # Cuenta los alérgenos separados por coma ignorando valores nulos
    df['allergens_n'] = df['allergens'].fillna('').apply(
        lambda x: len([a for a in x.split(',') if a.strip()])
    )

resumen_alergenos = df.groupby('categoria_macro')['allergens_n'].agg(
    total_alergenos='sum',
    promedio_por_producto='mean',
    productos_con_alergenos=lambda x: (x > 0).sum()
).sort_values(by='total_alergenos', ascending=False)

print("--- RESUMEN DE ALÉRGENOS POR CATEGORÍA MACRO ---")
print(resumen_alergenos)

In [ ]:

df['nutriscore_malo'] = (df['nutriscore_num'] <= 2).astype(float)

columnas_modelo = ['nova_group', 'allergens_n', 'categoria_macro', 'nutriscore_malo']
df_4c = df[columnas_modelo].dropna().copy()

# Codificación One-Hot para categoria_macro (drop_first=True evita multicolinealidad)
X_cat = pd.get_dummies(df_4c['categoria_macro'], drop_first=True, dtype=float)

# Definir predictores y variable objetivo
X = df_4c[['nova_group', 'allergens_n']].astype(float).join(X_cat)
Y = df_4c['nutriscore_malo']

X = sm.add_constant(X)

modelo_logistico = sm.Logit(Y, X).fit()

print(modelo_logistico.summary())

In [ ]:
# Agrupar 'Suplementos y Aditivos' dentro de 'Otros'
df_4c['categoria_macro'] = df_4c['categoria_macro'].replace({'Suplementos y Aditivos': 'Otros'})

X_cat = pd.get_dummies(df_4c['categoria_macro'], drop_first=True, dtype=float)
X = df_4c[['nova_group', 'allergens_n']].astype(float).join(X_cat)
X = sm.add_constant(X)

modelo_logistico_2 = sm.Logit(Y, X).fit()
print(modelo_logistico_2.summary())

In [ ]:
comparacion = pd.DataFrame({
    'Indicador': ['Pseudo R²', 'LLR p-value', 'AIC', 'BIC'],
    'Modelo 1': [
        modelo_logistico.prsquared,
        modelo_logistico.llr_pvalue,
        modelo_logistico.aic,
        modelo_logistico.bic
    ],
    'Modelo 2': [
        modelo_logistico_2.prsquared,
        modelo_logistico_2.llr_pvalue,
        modelo_logistico_2.aic,
        modelo_logistico_2.bic
    ]
})

display(comparacion)

In [ ]:

df_4c['categoria_macro'] = df_4c['categoria_macro'].replace({
    'Caldos y Sopas': 'Bebidas',
    'Lácteos': 'Bebidas'
})

X_cat = pd.get_dummies(df_4c['categoria_macro'], drop_first=True, dtype=float)

X = df_4c[['nova_group', 'allergens_n']].astype(float).join(X_cat)
X = sm.add_constant(X)
Y = df_4c['nutriscore_malo']

modelo_logistico_3 = sm.Logit(Y, X).fit()
print(modelo_logistico_3.summary())

# Analisis para las hipotesis

In [ ]:
import pandas as pd

df = pd.read_parquet("dataset_limpio.parquet")

print(df.columns.tolist())
df.info()

Empezaremos viendo posibles correlaciones lineales entre las variables 


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

variables_corr = [
    'energy_kcal',
    'fat_100g',
    'saturated_fat_100g',
    'carbs_100g',
    'sugars_100g',
    'fiber_100g',
    'proteins_100g',
    'salt_100g',
    'nutriscore_num',
    'ecoscore_num',
    'nova_group',
    'sodium_100g'
]

corr_spearman = df[variables_corr].corr(method='spearman')
corr_pearson = df[variables_corr].corr(method='pearson')

# Extraer el triángulo superior de la matriz para evitar pares duplicados (A-B y B-A) y la diagonal (A-A)
upper_tri_spearman = corr_spearman.where(np.triu(np.ones(corr_spearman.shape), k=1).astype(bool))

# Convertir a formato tabla
corr_pairs_spearman = upper_tri_spearman.stack().reset_index()
corr_pairs_spearman.columns = ['Variable_1', 'Variable_2', 'Correlacion']

# Filtrar por valor absoluto > 0.75 y ordenar
high_corr_spearman = corr_pairs_spearman[corr_pairs_spearman['Correlacion'].abs() > 0.70].sort_values(by='Correlacion', key=abs, ascending=False)

plt.figure(figsize=(12, 9))

sns.heatmap(
    corr_spearman,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f',
    linewidths=0.5
)

plt.title('Matriz de correlaciones - Spearman')
plt.tight_layout()
plt.show()


print("---- spearman -----")
print(high_corr_spearman)

Podemos observar que el sodio y las sales van muy de la mano, al igual que las grasas y las grasas saturadas. Esto es información redundante que solo añade ruido al modelo, por lo que solo tendremos en cuenta sales y grasas

In [ ]:

df.columns = df.columns.str.replace("\xa0", "").str.strip()

df = df.drop(["sodium_100g", "saturated_fat_100g"], axis=1, errors="ignore")

Procederemos a hacer un PCA para poder analizar relaciones en una menor dimension, entre nuestras nuevas 7 variables de estudio

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

variables_nutricionales = [
    'energy_kcal', 'fat_100g', 'carbs_100g', 'sugars_100g', 
    'fiber_100g', 'proteins_100g', 'salt_100g'
]

df_pca = df[variables_nutricionales].dropna().copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca)

pca = PCA(n_components=5)
componentes = pca.fit_transform(X_scaled)

varianza_explicada = pca.explained_variance_ratio_

print('Varianza explicada por componente:')
print(varianza_explicada)

print('\nVarianza acumulada:')
print(varianza_explicada.cumsum())

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=variables_nutricionales,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)],
)

print(loadings)

Vemos que la combinación lineal de los 3 primeros componentes esta muy distrubuida entre varias variables y recien con 4 componentes llegamos a una buena cifra de varianza explicada; y además esta 4ta componente esta prácticamente explicada por el coeficiente de las sales

In [ ]:
# graficamos para ver mejor las relaciones
pca = PCA(n_components=2)
componentes = pca.fit_transform(X_scaled)

loadings = pca.components_

plt.figure(figsize=(10, 8))

plt.scatter(
    componentes[:, 0],
    componentes[:, 1],
    alpha=0.4,
    color="steelblue",
    edgecolors="none",
    label="Observaciones",
)

scale_factor = np.max(np.abs(componentes[:, :2])) * 0.8

for i, variable in enumerate(variables_nutricionales):
    x_vec = loadings[0, i] * scale_factor
    y_vec = loadings[1, i] * scale_factor

    plt.arrow(
        0,
        0,
        x_vec,
        y_vec,
        color="crimson",
        alpha=0.8,
        width=scale_factor * 0.002,
        head_width=scale_factor * 0.03,
        head_length=scale_factor * 0.03,
        length_includes_head=True,
    )

    plt.text(
        x_vec * 1.15,
        y_vec * 1.15,
        variable,
        color="darkred",
        weight="bold",
        fontsize=10,
        ha="center",
        va="center",
    )

var_exp = pca.explained_variance_ratio_
plt.xlabel(f"PC1 ({var_exp[0]*100:.1f}% varianza)")
plt.ylabel(f"PC2 ({var_exp[1]*100:.1f}% varianza)")
plt.title(
    "Biplot de PCA (Componentes Principales y Vectores de Variables)",
    fontsize=12,
    weight="bold",
)

plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
plt.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()

plt.show()

sabemos que al hacer combinaciones lineales, aquellas variables que forman angulos rectos son aquellas que no están relacionadas entre sí. En este caso parece que tenemos 4 dimensiones independientes:
- azucares y carbohidratos
- fibras
- proteinas, energia_kilocalorias y grasas
- sales
(todas cada 100 gramos)

En este punto nos parece intersante también poder ver otro tipo de relaciones, por ejemplo, sabemos que en la realidad los productos con soja suelen tener un impacto ambiental negativo, por lo que queremos ver si nos encontramos con algo de eso en los datos

In [ ]:
df_eco_soja = df[['ecoscore_num', 'contains_soy']].copy()

df_eco_soja = df_eco_soja.dropna()

print("Registros disponibles:", len(df_eco_soja))

In [ ]:
tabla = pd.crosstab(
    df_eco_soja['contains_soy'],
    df_eco_soja['ecoscore_num']
)

tabla.index = ['Sin soja', 'Con soja']

display(tabla)

In [ ]:
tabla_porcentajes = pd.crosstab(
    df_eco_soja['contains_soy'],
    df_eco_soja['ecoscore_num'],
    normalize='index'
) * 100
tabla_porcentajes.index = ['Sin soja', 'Con soja']

tabla_porcentajes.plot(
    kind='bar',
    stacked=True,
    figsize=(9, 5)
)

plt.title('Distribución de Eco-Score según presencia de soja')
plt.xlabel('Presencia de soja')
plt.ylabel('Porcentaje de productos')
plt.legend(title='Eco-Score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(10, 6))

soja_por_nova = df.groupby('nova_group')['contains_soy'].mean() * 100

sns.barplot(
    x=soja_por_nova.index, 
    y=soja_por_nova.values, 
    hue=soja_por_nova.index, 
    palette='Reds', 
    legend=False,
    ax=ax
)

ax.set_title('Frecuencia de Soja vs Nivel NOVA')
ax.set_xlabel('Grupo NOVA (1: Mínimo -> 4: Ultraprocesado)')
ax.set_ylabel('% de Productos con Soja')

plt.tight_layout()
plt.show()

Sin embargo,a este punto parece que estamos llegando a conclusiones muy estrictas y no estamos viendo verdaderas relaciones. Por lo que una buena idea es plantearse otra estrategia que nos permita ver más relaciones, como un clusteing. Para ello necesitamos saber cual puede ser un buen número de cluster, por lo que haremos el gráfico del codo

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inercia = []
rango_k = range(1, 8)

for k in rango_k:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans_temp.fit(X_scaled)
    inercia.append(kmeans_temp.inertia_)

plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 6))
plt.plot(rango_k, inercia, marker='o', linestyle='-', color='#e74c3c', markersize=8)
plt.title('Método del Codo para el Dataset Completo')
plt.xlabel('Número de Clústeres (k)')
plt.ylabel('Inercia (Suma de distancias al cuadrado)')
plt.xticks(rango_k)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

df_model = df.copy()

n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
df_model['cluster'] = kmeans.fit_predict(X_scaled)


tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_results = tsne.fit_transform(X_scaled)

df_model['tsne_1'] = tsne_results[:, 0]
df_model['tsne_2'] = tsne_results[:, 1]


plt.style.use('seaborn-v0_8-whitegrid')

plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
sns.scatterplot(
    data=df_model, x='tsne_1', y='tsne_2', 
    hue='cluster', palette='Set1', alpha=0.7, s=30
)
plt.title('Proyección t-SNE coloreada por Clúster')

plt.subplot(1, 2, 2)
sns.scatterplot(
    data=df_model, x='tsne_1', y='tsne_2', 
    hue='nutriscore_num', palette='coolwarm', alpha=0.7, s=30
)
plt.title('Proyección t-SNE coloreada por su grado de nutricion')
plt.tight_layout()
plt.show()

Gracias a esto obervamos que parecieran exitir algunas relaciones entre los diferentes productos y el grado de nutrición de cada uno, como ocurre con el cluster de la esquina inferior derecha (donde vemos un conjunto de alimentos de características similares que son poco nutritivos). Sin embargo, no se nos deja muy en claro cual es su relación, debido a que tenemos productos de muchos tipos. Es por esto que vamos a intentar buscar relaciones entre productos de un mismo tipo usando la columna categoria_macro que generamos en la limpieza

In [ ]:
import math

def visualizar_tsne_nutricional(df, categoria_macro, variables_nutricionales, cant_clusters, perplexidad):
    df_cat = df[df['categoria_macro'] == categoria_macro].copy()
    n_muestras = len(df_cat)
    
    if n_muestras < 15:
        print(f"La categoría '{categoria_macro}' tiene pocas muestras ({n_muestras}).")
        return
        
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_cat[variables_nutricionales])
    
    kmeans = KMeans(n_clusters=cant_clusters, random_state=42, n_init='auto')
    df_cat['cluster'] = kmeans.fit_predict(X_scaled)
    df_cat['cluster_label'] = df_cat['cluster'].apply(lambda x: f"Clúster {x}")
    
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexidad)
    tsne_res = tsne.fit_transform(X_scaled)
    df_cat['tsne_1'] = tsne_res[:, 0]
    df_cat['tsne_2'] = tsne_res[:, 1]
    
    graficos_a_dibujar = ['cluster_label'] + variables_nutricionales
    n_graficos = len(graficos_a_dibujar)
    
    columnas = 4 
    filas = math.ceil(n_graficos / columnas)
    
    paletas = {
        'cluster_label': 'Set1', 
        'energy_kcal': 'inferno',  
        'fat_100g': 'YlOrBr',        
        'carbs_100g': 'Purples',    
        'sugars_100g': 'Blues',      
        'fiber_100g': 'Greens',
        'proteins_100g': 'Reds',     
        'salt_100g': 'bone_r',       
        'nutriscore_num': 'coolwarm' 
    }
    
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(filas, columnas, figsize=(columnas * 5, filas * 5))
    axes = axes.flatten() # Aplanar matriz para iterar fácilmente
    
    fig.suptitle(f"Perfil Espacial (t-SNE): {categoria_macro}", fontsize=20, weight='bold', y=1.02)

    for idx, var in enumerate(graficos_a_dibujar):
        color_map = paletas.get(var, 'viridis')
        
        sns.scatterplot(
            data=df_cat, x='tsne_1', y='tsne_2', 
            hue=var, palette=color_map, s=40, alpha=0.8, ax=axes[idx]
        )
        
        titulo = "Clústeres KMeans" if var == 'cluster_label' else var.replace('_100g', ' (g)').replace('_', ' ').title()
        axes[idx].set_title(titulo, fontsize=14, weight='bold')
        axes[idx].set_xlabel('')
        axes[idx].set_ylabel('')

    for vacio_idx in range(n_graficos, len(axes)):
        axes[vacio_idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
visualizar_tsne_nutricional(df, "Bebidas", variables_nutricionales, 4, 80)

In [ ]:
visualizar_tsne_nutricional(df, "Snacks y Dulces", variables_nutricionales, 4, 80)

Con estos gráficos podemos llegar a apreciar ciertas relaciones, que nos sugieren algunas hipótesis que podríamos plantear
- Bebidas: Existe un segmento de bebidas caracterizado por una elevada concentración de azúcares y carbohidratos, con un aporte reducido de proteínas y fibra, que presenta un perfil nutricional desfavorable frente a otros tipos de bebidas. (poco valor de mercado)
- Snacks: Dentro de los snacks existen productos con un aporte elevado de proteínas o fibra que, simultáneamente, presentan concentraciones importantes de grasas y energía, por lo que un perfil nutricional aparentemente favorable en ciertos nutrientes no necesariamente implica una mejor calidad nutricional global. (mejor valor de mercado)

Volamos a analizar el caso de la soja: 

In [ ]:
variables_nutricionales = [
    'energy_kcal', 'fat_100g', 'carbs_100g', 'sugars_100g', 
    'fiber_100g', 'proteins_100g', 'salt_100g'
]
variables_contexto = ['contains_soy', 'nova_group', 'nutriscore_num', 'ecoscore_num']

columnas_totales = variables_nutricionales + variables_contexto
df_clean = df.dropna(subset=columnas_totales).copy()

print(f"Total de productos tras la limpieza: {len(df_clean)}")

In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[variables_nutricionales])

kmeans = KMeans(n_clusters=4, random_state=25, n_init=50)
df_clean['cluster'] = kmeans.fit_predict(X_scaled)

tsne = TSNE(n_components=2, perplexity=40, random_state=52)
X_tsne = tsne.fit_transform(X_scaled)
df_clean['tsne_1'] = X_tsne[:, 0]
df_clean['tsne_2'] = X_tsne[:, 1]

print("Clustering y reducción de dimensionalidad completados.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

axes = axes.flatten()

sns.scatterplot(
    data=df_clean, x='tsne_1', y='tsne_2', 
    hue='cluster', palette='tab10', alpha=0.6, s=25, ax=axes[0]
)
axes[0].set_title('1. Mapa de Clusters Nutricionales')
axes[0].set_xlabel('Dimensión 1')
axes[0].set_ylabel('Dimensión 2')


scatter_eco = axes[1].scatter(
    df_clean['tsne_1'], df_clean['tsne_2'], 
    c=df_clean['ecoscore_num'], cmap='RdYlGn', alpha=0.8, s=25
)
axes[1].set_title('2. Impacto Ambiental (Eco-Score Numérico)')
plt.colorbar(scatter_eco, ax=axes[1], label='Eco-Score (Verde = Mejor)')
axes[1].set_xlabel('Dimensión 1')
axes[1].set_ylabel('Dimensión 2')

sns.scatterplot(
    data=df_clean, x='tsne_1', y='tsne_2', 
    hue='nova_group', palette='OrRd', alpha=0.8, s=25, ax=axes[2]
)
axes[2].set_title('3. Nivel de Procesamiento (Grupo NOVA)')
axes[2].legend(title='NOVA (4 = Ultraprocesado)')
axes[2].set_xlabel('Dimensión 1')
axes[2].set_ylabel('Dimensión 2')

sns.scatterplot(
    data=df_clean[~df_clean['contains_soy']], 
    x='tsne_1', y='tsne_2', color='lightgrey', alpha=0.3, s=20, label='Sin Soja', ax=axes[3]
)
sns.scatterplot(
    data=df_clean[df_clean['contains_soy']], 
    x='tsne_1', y='tsne_2', color='red', alpha=0.9, s=35, label='Con Soja', ax=axes[3]
)
axes[3].set_title('4. Presencia de Soja (Rojo)')
axes[3].legend()
axes[3].set_xlabel('Dimensión 1')
axes[3].set_ylabel('Dimensión 2')

plt.tight_layout()
plt.show()

Esto nos podría llegar a sugerir una posible separación de productos donde la soja es solo un aditivo vegetal, y otros donde se ultra procesa a fin de sacarle el máximo partido a nivel económico, a costa de un impacto ambiental mayor. Sin embargo, tampoco hay que toamar esto a la ligera, sino que es una posible conclusión (hipótesis) a demostrar